This notebook explores whether picking the home team is a good predictor of the winning team.

Questions we try to answer:
1. What percentage of games did the home team win (leave out the neutral games)? How close was your 60% guess?
2. On average, how many points does the home team score compared with the away team?
3. Which team had the best home record, and which had the worst?

In [120]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/games.csv")

In [121]:
# Store GAME IDs of "neutral" games
df["_is_home"] = df["MATCHUP"].str.contains("vs.", regex=False)

# Group by GAME_ID and sum "is_home"; filter for rows where sum is 0
home_counts = df.groupby("GAME_ID")["_is_home"].sum()

neutral_game_ids = home_counts[home_counts==0].index
neutral_game_ids

Index([22400147, 22400621, 22400633, 22401229, 22401230], dtype='int64',
      name='GAME_ID')

In [122]:
# Create `HOST_TEAM` column
parts = df["MATCHUP"].str.split(" ") # Split matchup into parts

df["HOST_TEAM"] = np.where(
    df["MATCHUP"].str.contains("@", regex=False),
    parts.str[-1], # NYK @ BOS --> BOS
    parts.str[0]   # BOS vs. NYK --> BOS
)

# Create `IS_HOME` column
df["IS_HOME"] = df["HOST_TEAM"] == df["TEAM_ABBREVIATION"]

In [123]:
# Check: should have 1230 rows
df["IS_HOME"].sum()


np.int64(1230)

In [124]:
# Check: one home row per game
df.groupby("GAME_ID")["IS_HOME"].sum().unique()

array([1])

In [125]:
# Split df to df_home and df_away; keep only important columns
important_cols = [
    "GAME_ID", 
    "GAME_DATE", 
    "TEAM_ABBREVIATION", 
    "PTS",
]
df_home = df[df["IS_HOME"] == True][important_cols]
df_away = df[df["IS_HOME"] == False][important_cols]

df_home.head()

,GAME_ID,GAME_DATE,TEAM_ABBREVIATION,PTS
0,22401187,2025-04-13,BOS,93
2,22401197,2025-04-13,SAS,125
6,22401196,2025-04-13,NOP,100
8,22401190,2025-04-13,MIA,118
9,22401189,2025-04-13,CLE,118


In [126]:
# Merge df_home and df_away on GAME_ID
df_merged = pd.merge(df_home, df_away, on=["GAME_ID", "GAME_DATE"], suffixes=["_home", "_away"])

df_merged = df_merged.rename(columns={
    "TEAM_ABBREVIATION_home": "HOME_TEAM", 
    "TEAM_ABBREVIATION_away": "AWAY_TEAM"}
    )

# Create `IS_NEUTRAL` column
df_merged["IS_NEUTRAL"] = df_merged["GAME_ID"].isin(neutral_game_ids)

In [127]:
# Check: 1230 rows
df_merged.shape

(1230, 7)

In [128]:
# Check: all games unique
df_merged["GAME_ID"].is_unique

True

In [129]:
# Check: home team never equal to away team
df_merged[df_merged["HOME_TEAM"] == df_merged["AWAY_TEAM"]]


,GAME_ID,GAME_DATE,HOME_TEAM,PTS_home,AWAY_TEAM,PTS_away,IS_NEUTRAL


In [130]:
# Check: IS_NEUTRAL sums to 5
df_merged["IS_NEUTRAL"].sum()

np.int64(5)

In [131]:
# Create `HOME_WIN` column
df_merged["HOME_WIN"] = df_merged["PTS_home"] > df_merged["PTS_away"]
df_merged["HOME_WIN"] = df_merged["HOME_WIN"].astype(int)



In [132]:
pct_home_win = df_merged[~df_merged["IS_NEUTRAL"]]["HOME_WIN"].mean()

print(f"Home team won {pct_home_win:.0%} of games - lower than my initial guess of 60%.")

Home team won 54% of games - lower than my initial guess of 60%.


In [133]:
df_merged["HOME_MARGIN"] = df_merged["PTS_home"] - df_merged["PTS_away"]
margin_home = df_merged[~df_merged["IS_NEUTRAL"]]["HOME_MARGIN"].mean()

print(f"On average, the home team scores {margin_home:.1f} points more than the away team.")

On average, the home team scores 1.7 points more than the away team.


In [134]:
sorted_home_pct = (
    df_merged.loc[~df_merged["IS_NEUTRAL"]]
    .groupby("HOME_TEAM")["HOME_WIN"]
    .mean()
    .sort_values(ascending=False)
    )

top_3 = sorted_home_pct.head(3)
bottom_3 = sorted_home_pct.tail(3)

print(f"The top three teams with the best home record are {top_3.index[0]}, {top_3.index[1]}, and {top_3.index[2]} with win percentages of {top_3.iloc[0]:.0%}, {top_3.iloc[1]:.0%}, and {top_3.iloc[2]:.0%}")
print(f"The bottom three teams with the worst home record are {bottom_3.index[-1]}, {bottom_3.index[-2]}, and {bottom_3.index[-3]} with win percentages of {bottom_3.iloc[-1]:.0%}, {bottom_3.iloc[-2]:.0%}, and {bottom_3.iloc[-3]:.0%}")

The top three teams with the best home record are OKC, CLE, and LAL with win percentages of 85%, 83%, and 76%
The bottom three teams with the worst home record are WAS, UTA, and CHA with win percentages of 20%, 24%, and 29%


If we predicted "home team wins" for every game, we would only be 54% accurate: just slightly better than flipping a coin. This is our baseline.

In [135]:
df_merged.to_csv("data/one-row-per-game.csv", index=False)